# Multi-agent research pipeline on a Kaggle GPU — Nemotron Nano v2 (`huggingface` backend)

Runs the full six-agent pipeline (Literature → Hypothesis → Experiment Planner → Coder → Writer ⇄ Reviewer)
end to end inside a Kaggle notebook, driven by **NVIDIA Nemotron Nano v2** loaded *in-process* with
transformers — no server, no port, no build step.

**No pipeline code changes are involved.** This is the repo's existing `LLM_BACKEND=huggingface` path
(see [`llm.py`](https://github.com/<you>/<this-repo>/blob/main/src/research_pipeline/llm.py)) — the same one
[`scripts/slurm/run_pipeline_hf.sbatch`](https://github.com/<you>/<this-repo>/blob/main/scripts/slurm/run_pipeline_hf.sbatch)
uses on Barkla. It's the sibling of
[`kaggle_gemma4_pipeline.ipynb`](kaggle_gemma4_pipeline.ipynb), which instead serves a 4-bit GGUF quant
through `llama-server` (the `openai` backend) — pick whichever notebook matches the backend you want to
exercise. This one has no llama.cpp CUDA build, at the cost of a larger download and more VRAM.

### Model

[`nvidia/NVIDIA-Nemotron-Nano-12B-v2`](https://huggingface.co/nvidia/NVIDIA-Nemotron-Nano-12B-v2) — a hybrid
Mamba-2 + Transformer *reasoning* model (mostly Mamba-2/MLP layers, four attention layers), ~24GB at
BF16/FP16. `trust_remote_code=True` is already hardcoded in `llm.py`, and transformers' native `nemotron_h`
modeling code has a pure-PyTorch fallback for the Mamba-2 scan, so — unlike llama.cpp for Gemma — nothing
here needs to compile a CUDA kernel (`mamba-ssm`/`causal-conv1d`); it's just slower per token without them.

**Reasoning mode.** Like Nemotron 3 Nano, this model thinks before answering. The `huggingface` backend has
no request body to attach `chat_template_kwargs.enable_thinking` to (that's an `openai`-backend-only wire
field — see `llm.py`), so `LLM_ENABLE_THINKING` is logged and ignored here; `llm_json.strip_reasoning` is the
*only* guard against a `<think>` trace breaking a JSON parse, which is fine — it's exactly the case it was
written for.

**VRAM.** ~24GB of weights alone exceeds a single 16GB Kaggle T4 or P100. `HF_DEVICE_MAP="auto"` (below)
lets `accelerate` shard the model across every GPU the process can see, so this notebook needs the
**GPU T4 x2** accelerator (2×16GB = 32GB) for headroom; a single-GPU accelerator will very likely
`CUDA out of memory`. If only one GPU is available, switch `MODEL_ID` to the smaller
[`nvidia/NVIDIA-Nemotron-Nano-9B-v2`](https://huggingface.co/nvidia/NVIDIA-Nemotron-Nano-9B-v2) (~18GB) — still
tight on a single 16GB card, but closer.

### Kaggle settings this notebook requires

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | GPU T4 x2 | ~24GB of weights need both cards' VRAM; a single T4/P100 will likely OOM |
| **Internet** | **On** | arXiv, Semantic Scholar, Hugging Face, PyPI, GitHub |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` outputs between sessions |

Expect roughly **10–20 min** of one-time setup (installing torch/transformers, then a ~24GB weights
download) before the first agent runs, plus a few minutes each time an agent loads its own copy of the
model (see section 4).

## 0. Configuration

Everything you'd normally want to change lives in this one cell.

In [ ]:
# --- model -------------------------------------------------------------------
MODEL_ID = "nvidia/NVIDIA-Nemotron-Nano-12B-v2"   # hybrid Mamba-2/Transformer, ~24GB at bf16/fp16.
                                                    # Smaller alt for a single GPU: "nvidia/NVIDIA-Nemotron-Nano-9B-v2" (~18GB)
HF_DEVICE_MAP = "auto"     # accelerate shards the model across every GPU this process can see
HF_DTYPE      = "float16"  # Kaggle's T4/P100 are pre-Ampere (no bf16 support) - use "bfloat16" on an Ampere+ card

# --- the run -------------------------------------------------------------------
RESEARCH_QUESTION = "How can retrieval-augmented generation reduce hallucination in scientific summarization?"
MAX_RESULTS_PER_QUERY = 5   # papers per generated query, per source (arXiv + Semantic Scholar)
MAX_ITERATIONS        = 2   # Writer/Reviewer cycles; each one is a full paper redraft
QUALITY_THRESHOLD     = 4   # 1-5, the Reviewer's per-dimension pass mark

# --- where the code comes from --------------------------------------------------
# "git"     : clone from a GitHub remote (set REPO_URL)
# "dataset" : you uploaded the repo as a Kaggle Dataset (set REPO_PATH to its dir)
REPO_SOURCE = "git"
REPO_URL    = "https://github.com/<you>/<this-repo>.git"
REPO_PATH   = "/kaggle/input/<your-dataset-slug>"

# --- optional secrets -----------------------------------------------------------
# Add via Kaggle's "Secrets" panel (Add-ons -> Secrets) rather than pasting keys here.
SEMANTIC_SCHOLAR_SECRET = "SEMANTIC_SCHOLAR_API_KEY"   # skipped, with a warning, if unset
HF_TOKEN_SECRET          = "HF_TOKEN"                  # only needed if MODEL_ID ever requires a license acceptance

WORK_DIR     = "/kaggle/working"        # outputs land here and become notebook output
SCRATCH_DIR  = "/kaggle/temp"           # not committed as notebook output
HF_CACHE_DIR = f"{SCRATCH_DIR}/hf-cache"  # ~24GB of weights - keep off /kaggle/working, which is size-capped

## 1. Probe the machine

Check the accelerator and network before installing several GB of torch/transformers.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

print("GPU:")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "  none detected - Settings -> Accelerator -> GPU T4 x2")
print("disk free: %.1f GB" % (shutil.disk_usage("/").free / 1e9))

import socket
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    print("internet: ok")
except OSError:
    print("internet: BLOCKED - turn Internet on in the notebook Settings panel and re-run")

## 2. Get the pipeline and install it

Kaggle has no `uv` and the repo isn't on PyPI, so this installs the package from source with pip, pulling in
the `huggingface` extra (`torch`, `transformers`, `accelerate`, `langchain-huggingface` — several GB). `uv`
is installed too — not for this install, but because the **Coder Agent uses `uv venv` at runtime** to build
an isolated environment whenever generated experiment code needs a package that isn't already importable.
Without it on `PATH`, those experiments come back as `code_generated_not_run` with a "'uv' is not on PATH"
reason instead of results.

In [ ]:
import importlib, os, shutil, subprocess, sys
from pathlib import Path

if REPO_SOURCE == "git":
    repo_dir = Path("/kaggle/working/research-pipeline")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
else:
    # /kaggle/input is read-only, and an editable install needs to write egg-info.
    src = Path(REPO_PATH)
    repo_dir = Path("/kaggle/working/research-pipeline")
    if not repo_dir.exists():
        shutil.copytree(src, repo_dir)

print("repo:", repo_dir)
assert (repo_dir / "pyproject.toml").exists(), f"no pyproject.toml under {repo_dir}"

# [huggingface], not the bare package - that extra is what pulls in torch/transformers/accelerate.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[huggingface]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# pip installs console scripts next to the interpreter; the Coder Agent looks for
# `uv` on PATH, which in a notebook kernel doesn't include that directory.
os.environ["PATH"] = f"{Path(sys.executable).parent}:{os.environ['PATH']}"
print("uv:", shutil.which("uv"))

# An editable install is a .pth/finder file that Python's site module only
# processes at interpreter startup - a plain `import` right after `pip install -e`
# in the SAME live kernel won't see it (ModuleNotFoundError) until the kernel
# restarts. Adding src/ to sys.path directly sidesteps that timing problem
# entirely, so this notebook never needs a restart mid-run. This is the one
# path that matters here - pip's own dependency installs (torch, transformers,
# langgraph, etc.) land in site-packages, which is already on sys.path.
sys.path.insert(0, str(repo_dir / "src"))
importlib.invalidate_caches()

import research_pipeline
print("research_pipeline:", research_pipeline.__file__)

## 3. Configure the pipeline

**Order matters here.** `research_pipeline.config` reads the environment at *import* time and freezes it
into a module-level `settings` object, so every variable has to be set before the first import of a
`research_pipeline` submodule that touches `config` (e.g. `research_pipeline.llm`). If you re-run this cell
after that first import, restart the kernel — editing `os.environ` afterwards changes nothing.

Under `LLM_BACKEND=huggingface`, `LLM_MODEL` is a Hugging Face repo id passed straight to
`AutoModelForCausalLM.from_pretrained` (via `HuggingFacePipeline.from_model_id`) — not a server alias like
in the GGUF notebook, since there is no server.

In [ ]:
import os
from pathlib import Path

os.chdir(WORK_DIR)   # relative output dirs resolve here
os.environ["HF_HOME"] = HF_CACHE_DIR   # keep the ~24GB weights cache off /kaggle/working

# --- model (in-process, no server) -------------------------------------------
os.environ["LLM_BACKEND"]       = "huggingface"
os.environ["LLM_MODEL"]         = MODEL_ID
os.environ["LLM_HF_DEVICE_MAP"] = HF_DEVICE_MAP
os.environ["LLM_HF_DTYPE"]      = HF_DTYPE

os.environ["LLM_TEMPERATURE"] = "0.2"
os.environ["LLM_TOP_P"]       = "0.95"
# Truncation is a hard failure (a half-written Writer section never parses), so leave room.
os.environ["LLM_MAX_TOKENS"]  = "8192"
# No-op under this backend (see cell above / llm.py) - kept for parity with the openai-backend
# notebook and because llm_json.strip_reasoning is the guard that actually matters here.
os.environ["LLM_ENABLE_THINKING"] = "false"

# --- optional secrets ---------------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

if secrets is not None:
    try:
        os.environ["SEMANTIC_SCHOLAR_API_KEY"] = secrets.get_secret(SEMANTIC_SCHOLAR_SECRET)
        print("Semantic Scholar key loaded from Kaggle Secrets")
    except Exception as exc:
        print(f"No Semantic Scholar key ({exc.__class__.__name__}) - that source is skipped, arXiv still runs")
    try:
        from huggingface_hub import login
        login(secrets.get_secret(HF_TOKEN_SECRET))
        print("Hugging Face token loaded from Kaggle Secrets")
    except Exception as exc:
        print(f"No Hugging Face token ({exc.__class__.__name__}) - fine unless {MODEL_ID} turns out to be gated")

# --- where artifacts go ---------------------------------------------------------
outputs = Path(WORK_DIR) / "outputs"
for var in ("HYPOTHESIS_OUTPUT_DIR", "EXPERIMENT_PLANNER_OUTPUT_DIR",
            "CODER_OUTPUT_DIR", "WRITER_OUTPUT_DIR", "REVIEWER_OUTPUT_DIR"):
    os.environ[var] = str(outputs)
os.environ["WRITER_REVIEWER_LOOP_OUTPUT_DIR"] = str(outputs / "paper")
os.environ["CODER_EXPERIMENTS_DIR"] = str(Path(WORK_DIR) / "experiments")

# There is no SLURM here, so nothing may try to submit a job. This is already the
# default; set explicitly because it's the one setting that would fail loudly on Kaggle.
os.environ["CODER_AUTO_SUBMIT_SLURM"] = "false"

os.environ["WRITER_REVIEWER_MAX_ITERATIONS"]    = str(MAX_ITERATIONS)
os.environ["WRITER_REVIEWER_QUALITY_THRESHOLD"] = str(QUALITY_THRESHOLD)

print("configured for", os.environ["LLM_MODEL"], "on", os.environ["LLM_HF_DEVICE_MAP"], f"({HF_DTYPE})")

### 3b. Load the model and smoke-test it

There's no server to health-check here, so this loads the model directly through the same
`get_chat_model()` factory every agent uses, and sends it one request. That turns the first (multi-minute)
weight load and any architecture/VRAM problem into a clear failure right here, instead of a confusing one
buried inside the Literature agent five minutes into section 4.

In [ ]:
import time

from research_pipeline.llm import get_chat_model

print(f"loading {os.environ['LLM_MODEL']} onto {os.environ['LLM_HF_DEVICE_MAP']} ({HF_DTYPE}) "
      "- first call, downloads on first run, several minutes either way...")
started = time.time()
llm = get_chat_model(temperature=0.0)
response = llm.invoke('Reply with only this JSON: {"ok": true}')
print(f"loaded + first response in {(time.time() - started) / 60:.1f} min")
print(response.content)

# Each agent below builds its own copy of the model (get_chat_model() isn't cached across
# callers), so this smoke-test copy is freed here rather than left holding VRAM for the run.
import gc
import torch

del llm, response
gc.collect()
torch.cuda.empty_cache()

## 4. Run the pipeline

One `graph.stream` — the same call `research-pipeline orchestrate` makes. Stages stream as they complete
rather than reporting only at the end, so a long run is observable.

**What to expect on Kaggle.** A GPU *is* present, so the Coder Agent runs `low`/`medium` complexity
experiments synchronously, including ones that self-report `needs_gpu` — the probe is at runtime, not
hardcoded. `high` complexity plans still get code plus a `run.sbatch` they'll never be submitted with here;
that file is a harmless artifact on a machine with no scheduler.

**Model loads happen per agent, not per LLM call.** Each of the six agents constructs its own chat model
once (inside its own `__init__`/node function) and reuses it for every call it makes, so expect roughly six
more multi-minute "loading weights" pauses spread across this run — cheaper than the first one in section
3b since the files are already in `HF_CACHE_DIR`, but not free.

In [ ]:
import logging, time, uuid

logging.getLogger("research_pipeline").setLevel(logging.INFO)

from research_pipeline.orchestrator.graph import build_pipeline_graph

graph = build_pipeline_graph()
initial_state = {
    "research_question": RESEARCH_QUESTION,
    "max_results_per_query": MAX_RESULTS_PER_QUERY,
    "download_dir": f"{WORK_DIR}/papers",
    "metadata_path": f"{WORK_DIR}/papers/metadata.json",
    "output_dir": str(outputs),
    "max_iterations": MAX_ITERATIONS,
    "quality_threshold": QUALITY_THRESHOLD,
}
run_config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 50}

started = time.time()
final_state = {}
for update in graph.stream(initial_state, config=run_config, stream_mode="updates"):
    for node, delta in update.items():
        print(f"[{time.time() - started:7.1f}s] {node}: {sorted(delta)}")
        final_state.update(delta)

print(f"\ndone in {(time.time() - started) / 60:.1f} min")

## 5. Results

In [ ]:
result = final_state["final_result"]

print("final paper   :", result["final_paper_path"])
print("iterations run:", result["iterations_run"])
print("converged     :", result["converged"])
print("review history:", result["review_history_path"])
for issue in result["unresolved_issues"]:
    print("  unresolved:", issue)

# Per-experiment outcomes: "completed" ran here, "code_generated_not_run" did not.
for experiment in final_state.get("coder_output", {}).get("experiments", []):
    print(f"  {experiment['hypothesis_id']}: {experiment['status']} {experiment.get('reason', '')}".rstrip())

### Keep the artifacts

Everything already lives under `/kaggle/working`, so **Save Version** captures it. This bundles the run into
one archive to download instead of clicking through the output tree — the PDF, every review iteration, the
generated experiment code, and the downloaded papers.

In [ ]:
import shutil
from pathlib import Path

archive = shutil.make_archive(f"{WORK_DIR}/run_artifacts", "zip", root_dir=WORK_DIR, base_dir="outputs")
print(archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")

# FileLink, not an embedded IFrame: Jupyter resolves embeds relative to the
# notebook's serving root, and these paths are absolute.
from IPython.display import FileLink, display
display(FileLink(Path(archive).relative_to(WORK_DIR).as_posix()))
display(FileLink(Path(result["final_paper_path"]).resolve().relative_to(WORK_DIR).as_posix()))

## 6. Free the GPU

There's no server process to stop here, only Python references keeping weights resident. Do this before
running anything else GPU-heavy in the same session.

In [ ]:
import gc

import torch

del graph
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed (best effort)")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `No CUDA GPU detected` / empty GPU list in cell 1 | Accelerator is "None" | Settings → Accelerator → GPU T4 x2, then restart the session |
| `CUDA out of memory` in section 3b or during a later agent's first call | ~24GB of weights (+ activation/SSM-state memory) don't fit what this process can see | Use **GPU T4 x2**, not a single T4/P100; `HF_DEVICE_MAP="auto"` already shards across every visible GPU. If still tight, switch `MODEL_ID` to `nvidia/NVIDIA-Nemotron-Nano-9B-v2` |
| `ImportError: LLM_BACKEND=huggingface requires the 'huggingface' extra` | cell 2's install didn't include the extra | Re-run cell 2 — it installs `{repo_dir}[huggingface]`, not the bare package |
| Unrecognized model type / architecture error (e.g. mentioning `nemotron_h`) | Installed `transformers` predates NemotronH support | `pip install -q --upgrade transformers accelerate`, restart the kernel, re-run from cell 2 |
| `ModuleNotFoundError: No module named 'research_pipeline'` in cell 5 | An editable install's `.pth` finder is only processed by Python's `site` module at interpreter startup — a live kernel doesn't pick it up mid-session | Already handled — cell 2 adds `repo_dir/src` to `sys.path` directly instead of relying on the install to be re-scanned, and smoke-tests the import right there. If it still happens, cell 2 didn't actually run before cell 5 — re-run it |
| `401` / gated-repo error downloading the model | `MODEL_ID`'s Hugging Face page requires accepting a license | Accept it there, then add an `HF_TOKEN` Kaggle Secret (loaded automatically in cell 4 if present) |
| Download very slow, or fills the disk | ~24GB safetensors download into `HF_CACHE_DIR` | Check free space in cell 1; `SCRATCH_DIR` needs ~30GB free |
| `LLMJSONError` after the repair retry, or a `<think>`/`<SPECIAL_...>` tag leaking into output | Reasoning trace not fully stripped, or the model drifting off-format | `llm_json.strip_reasoning` should already catch the trace; if parses still fail, lower `LLM_TEMPERATURE` to `0.0` in cell 4 |
| `'uv' is not on PATH` in an experiment `reason` | Kernel `PATH` missing pip's script dir | Re-run cell 2, which prepends it |
| Semantic Scholar warning at startup | No API key | Expected — arXiv still runs; add the key via Add-ons → Secrets |
| Run outlives the session | Kaggle caps a GPU session at 9h (12h/week free quota); every agent's first call re-loads the model (a few minutes each) on top of inference | Lower `MAX_ITERATIONS`/`MAX_RESULTS_PER_QUERY`, or switch to the smaller 9B v2 |

### Running it from a terminal instead

```bash
pip install -e ".[huggingface]"
LLM_BACKEND=huggingface LLM_MODEL=nvidia/NVIDIA-Nemotron-Nano-12B-v2 \
  LLM_HF_DEVICE_MAP=auto LLM_HF_DTYPE=float16 \
  research-pipeline orchestrate "your research question"
```